<a href="https://colab.research.google.com/github/KJ0211/Machine-learning-portfolio/blob/main/Automated%20Essay%20Scoring%20with%20a%20Fine-Tuned%20Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install transformers datasets accelerate scikit-learn pandas evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00


In [2]:
import torch, transformers, datasets
print(torch.__version__, transformers.__version__)
print("GPU available:", torch.cuda.is_available())


2.11.0+cu128 5.16.1
GPU available: True


In [6]:
import pandas as pd

df = pd.read_csv("training_set_rel3.tsv", sep="\t", encoding="latin-1")
print(df.shape)
print(df.columns.tolist())
print(df[["essay_id", "essay_set", "domain1_score"]].head())
print(df["essay_set"].value_counts().sort_index())

(12976, 28)
['essay_id', 'essay_set', 'essay', 'rater1_domain1', 'rater2_domain1', 'rater3_domain1', 'domain1_score', 'rater1_domain2', 'rater2_domain2', 'domain2_score', 'rater1_trait1', 'rater1_trait2', 'rater1_trait3', 'rater1_trait4', 'rater1_trait5', 'rater1_trait6', 'rater2_trait1', 'rater2_trait2', 'rater2_trait3', 'rater2_trait4', 'rater2_trait5', 'rater2_trait6', 'rater3_trait1', 'rater3_trait2', 'rater3_trait3', 'rater3_trait4', 'rater3_trait5', 'rater3_trait6']
   essay_id  essay_set  domain1_score
0         1          1              8
1         2          1              9
2         3          1              7
3         4          1             10
4         5          1              8
essay_set
1    1783
2    1800
3    1726
4    1770
5    1805
6    1800
7    1569
8     723
Name: count, dtype: int64


In [7]:
SET_ID = 1
sub = df[df["essay_set"] == SET_ID][["essay_id", "essay", "domain1_score"]].dropna()
print(sub.shape)
print(sub["domain1_score"].min(), sub["domain1_score"].max())

(1783, 3)
2 12


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import cohen_kappa_score
import numpy as np

train_df, test_df = train_test_split(sub, test_size=0.2, random_state=42)

# dumbest possible baseline: word count -> score
X_train = train_df["essay"].str.split().str.len().values.reshape(-1, 1)
X_test = test_df["essay"].str.split().str.len().values.reshape(-1, 1)
y_train, y_test = train_df["domain1_score"].values, test_df["domain1_score"].values

baseline = LinearRegression().fit(X_train, y_train)
pred = np.clip(np.round(baseline.predict(X_test)), y_train.min(), y_train.max())

qwk = cohen_kappa_score(y_test, pred, weights="quadratic")
print("Baseline QWK:", qwk)

Baseline QWK: 0.7804065490393586


In [9]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"  # smaller/faster than bert-base; swap in bert-base-uncased later if you want
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = Dataset.from_pandas(train_df[["essay", "domain1_score"]].rename(columns={"domain1_score": "label"}))
test_ds = Dataset.from_pandas(test_df[["essay", "domain1_score"]].rename(columns={"domain1_score": "label"}))

def tokenize(batch):
    return tokenizer(batch["essay"], truncation=True, padding="max_length", max_length=512)

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/1426 [00:00<?, ? examples/s]

Map:   0%|          | 0/357 [00:00<?, ? examples/s]

In [10]:
example = train_ds[0]
print(tokenizer.decode(example["input_ids"], skip_special_tokens=True)[:300])
print("Original: ", train_df["essay"].iloc[0][:300])

computers and the @ caps1 were a technological break through. it exposed to the average world, things that were never thought possitive. but as these things advanced over the years, they ' ve become an addiction so bad of an addiction its begun to threaten peoples lives i ' ve been given a choice to
Original:  Computers and the @CAPS1 were a technological break through. It exposed to the average world, things that were never thought possitive. But as these things advanced over the years, they've become an addiction so bad of an addiction its begun to threaten peoples lives I've been given a choice to s'de


In [12]:
import datasets.config
datasets.config.TORCHVISION_AVAILABLE = False

In [14]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=1, problem_type="regression"
)

from datasets import Value

train_ds = train_ds.cast_column("label", Value("float32"))
test_ds = test_ds.cast_column("label", Value("float32"))

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.clip(np.round(preds.squeeze()), y_train.min(), y_train.max())
    return {"qwk": cohen_kappa_score(labels, preds, weights="quadratic")}

args = TrainingArguments(
    output_dir="./essay_scoring_model",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="qwk",
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

print(train_ds.features["label"])
trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Casting the dataset:   0%|          | 0/1426 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/357 [00:00<?, ? examples/s]

Value('float32')


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=716, training_loss=3.967193611507309, metrics={'train_runtime': 347.927, 'train_samples_per_second': 16.394, 'train_steps_per_second': 2.058, 'total_flos': 755580566986752.0, 'train_loss': 3.967193611507309, 'epoch': 4.0})

In [15]:
metrics = trainer.evaluate()
print(metrics)
print("Baseline QWK was:", qwk)

Training Loss,Validation Loss,Epoch,Qwk
0.758483,0.587989,4,0.855412


{'eval_loss': 0.5879886150360107, 'eval_qwk': 0.855412161911213}
Baseline QWK was: 0.7804065490393586
